# 00 · Definiciones

Catálogo de las funciones de `src/utils.py`: qué hace cada una, a qué concepto del
apunte corresponde, y una corrida sobre una **serie sintética** (tendencia +
estacionalidad semanal + ruido) para no depender todavía del dataset real.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import utils

# --- Serie sintética de ejemplo: tendencia + estacionalidad semanal + ruido ---
rng = np.random.default_rng(42)
n = 180
fechas = pd.date_range("2021-01-01", periods=n, freq="D")
t = np.arange(n)
tendencia = 1000 + 5 * t                          # sube de a poco
estacional = 300 * np.sin(2 * np.pi * t / 7)      # ciclo semanal (periodo 7 días)
ruido = rng.normal(0, 80, n)
serie = pd.Series(tendencia + estacional + ruido, index=fechas, name="streams")

utils.graficar_serie(serie, "Serie sintética de ejemplo (tendencia + semanal + ruido)")
plt.show()
serie.head()

C:\Users\micag\AppData\Local\Temp\ipykernel_2384\134281007.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


2021-01-01    1024.377366
2021-01-02    1156.350716
2021-01-03    1362.514469
2021-01-04    1220.410299
2021-01-05     733.752063
Freq: D, Name: streams, dtype: float64

## `cargar_dataset_completo(path_csv_grande)`

Carga el CSV completo de Spotify Charts (leído en chunks) y lo filtra a
`region in ["Argentina", "Global"]`. **Concepto:** adquisición/muestreo de la señal
— pasar de los datos crudos a la señal con la que vamos a trabajar.

In [2]:
import tempfile, os

# Mini-CSV de juguete con varias regiones para probar el filtrado.
demo = pd.DataFrame({
    "title": ["A", "B", "C", "D"],
    "rank": [1, 2, 1, 3],
    "date": pd.to_datetime(["2021-01-01"] * 4),
    "artist": ["x", "y", "z", "w"],
    "url": [""] * 4,
    "region": ["Argentina", "Global", "Chile", "Brazil"],
    "chart": ["top200"] * 4,
    "trend": ["SAME_POSITION"] * 4,
    "streams": [100, 200, 300, 400],
})
tmp = os.path.join(tempfile.gettempdir(), "demo_charts.csv")
demo.to_csv(tmp, index=False)

df_demo = utils.cargar_dataset_completo(tmp)
print("Regiones tras el filtro:", sorted(df_demo["region"].unique()))
df_demo

Regiones tras el filtro: ['Argentina', 'Global']


,title,rank,date,artist,url,region,chart,trend,streams
0,A,1,2021-01-01,x,NaN,Argentina,top200,SAME_POSITION,100
1,B,2,2021-01-01,y,NaN,Global,top200,SAME_POSITION,200


## `construir_serie(df, filtro)`

Arma una `pandas.Series` indexada por fecha a partir del DataFrame de charts, según
un `filtro` (title/artist/region/chart, columna de valor y forma de agregado).
**Concepto:** definición de la señal a analizar (una canción/artista como serie temporal).

In [3]:
df_ejemplo = pd.DataFrame({
    "title": ["Hit"] * 3 + ["Otra"],
    "artist": ["Banda"] * 3 + ["X"],
    "date": pd.to_datetime(["2021-01-01", "2021-01-02", "2021-01-03", "2021-01-01"]),
    "region": ["Argentina"] * 4,
    "chart": ["top200"] * 4,
    "streams": [100, 150, 130, 999],
})
s = utils.construir_serie(df_ejemplo, {"title": "Hit", "region": "Argentina", "valor": "streams"})
print(s)

date
2021-01-01    100
2021-01-02    150
2021-01-03    130
Name: streams, dtype: int64


## `interpolar_serie(serie)`

Reindexa a frecuencia diaria continua e interpola linealmente los días faltantes
(los "silencios" fuera del chart). **Concepto:** reconstrucción / muestreo uniforme
de la señal (condición necesaria para FFT y filtrado).

In [4]:
con_huecos = serie.drop(serie.index[[10, 11, 12, 40]])
interpolada = utils.interpolar_serie(con_huecos)
print(f"Con huecos: {len(con_huecos)} muestras | Interpolada: {len(interpolada)} muestras")
print("¿Índice diario continuo?", (interpolada.index.freq is not None) or
      (interpolada.index.to_series().diff().dropna() == pd.Timedelta("1D")).all())

Con huecos: 176 muestras | Interpolada: 180 muestras
¿Índice diario continuo? True


## `resumen_estadistico(serie)`

Media, desvío estándar y rango. **Concepto:** caracterización estadística de la
señal (valor medio, dispersión, excursión).

In [5]:
utils.resumen_estadistico(serie)

{'media': 1445.8867212551213,
 'desvio': 339.23704842135595,
 'rango': 1612.5294794567699}

## `graficar_serie(serie, titulo)`

Grafica la señal en el dominio del tiempo (valor vs. fecha). **Concepto:** forma de
onda de la señal.

In [6]:
utils.graficar_serie(serie, "Forma de onda")
plt.show()

C:\Users\micag\AppData\Local\Temp\ipykernel_2384\1292347612.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## `calcular_fft(serie)`

FFT real (`np.fft.rfft`) de la serie sin su media; devuelve frecuencias en
**ciclos/día** y magnitudes. **Concepto:** paso al dominio de la frecuencia
(análisis espectral de Fourier). La función solo quita la media (DC); como la
tendencia lineal mete mucha energía en baja frecuencia, acá le restamos primero
un ajuste lineal para que se vea el pico semanal.

In [7]:
# Quitamos la tendencia lineal para que el pico semanal domine el espectro.
coef = np.polyfit(t, serie.values, 1)
serie_sin_tendencia = serie - np.polyval(coef, t)

frecs, mags = utils.calcular_fft(serie_sin_tendencia)
f_pico = frecs[np.argmax(mags)]
print(f"Frecuencia dominante: {f_pico:.4f} ciclos/día -> período {1/f_pico:.2f} días (esperado ~7)")

Frecuencia dominante: 0.1444 ciclos/día -> período 6.92 días (esperado ~7)


## `graficar_espectro(frecuencias, magnitudes, titulo)`

Grafica el espectro de magnitud. **Concepto:** representación en frecuencia; el pico
en 1/7 día es la estacionalidad semanal.

In [8]:
utils.graficar_espectro(frecs, mags, "Espectro de la serie sintética")
plt.axvline(1/7, color="red", ls="--", lw=1, label="1/7 (semanal)")
plt.legend(); plt.show()

C:\Users\micag\AppData\Local\Temp\ipykernel_2384\2424290208.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.show()


## `filtro_media_movil(serie, ventana)`

Media móvil (MA) centrada. **Concepto:** filtro pasa-bajos en el tiempo (suaviza el
ruido de alta frecuencia).

In [9]:
ma7 = utils.filtro_media_movil(serie, 7)
utils.graficar_serie(serie, "Original vs. MA7")
plt.plot(ma7.index, ma7.values, color="orange", label="MA7"); plt.legend(); plt.show()

C:\Users\micag\AppData\Local\Temp\ipykernel_2384\3433455810.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.plot(ma7.index, ma7.values, color="orange", label="MA7"); plt.legend(); plt.show()


## `descomponer_MA7_MA21(serie)`

Descompone en tendencia (MA21), estacional (MA7 − MA21) y residuo (serie − MA7),
siguiendo el algoritmo del apunte. **Concepto:** separación tendencia /
estacionalidad / ruido vía filtrado.

In [10]:
desc = utils.descomponer_MA7_MA21(serie)
fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ax[0].plot(desc["tendencia"]); ax[0].set_title("Tendencia (MA21)")
ax[1].plot(desc["estacional"]); ax[1].set_title("Estacional (MA7 − MA21)")
ax[2].plot(desc["residuo"]); ax[2].set_title("Residuo (serie − MA7)")
plt.tight_layout(); plt.show()

C:\Users\micag\AppData\Local\Temp\ipykernel_2384\2694879116.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## `filtro_pasabanda_frecuencia(serie, f_low, f_high)`

Filtra en frecuencia: pone en cero el espectro fuera de `[f_low, f_high]` y vuelve
con `ifft`. **Concepto:** filtro pasa-banda ideal en el dominio de Fourier.

In [11]:
# Banda angosta alrededor de la frecuencia semanal (1/7 ≈ 0.143 ciclos/día)
banda = utils.filtro_pasabanda_frecuencia(serie, 0.12, 0.16)
plt.figure(figsize=(12, 4))
plt.plot(banda.index, banda.values, color="purple")
plt.title("Componente semanal aislada (pasa-banda ~1/7)")
plt.show()

C:\Users\micag\AppData\Local\Temp\ipykernel_2384\1366861261.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## `energia_serie(serie)`

Energía en el tiempo `Σx(n)²` y en frecuencia `Σ|X(k)|²/N`, con la diferencia
relativa entre ambas. **Concepto:** energía de la señal y teorema de Parseval.

In [12]:
e = utils.energia_serie(serie)
print(f"Energía (tiempo):     {e['energia_tiempo']:.4e}")
print(f"Energía (frecuencia): {e['energia_frecuencia']:.4e}")
print(f"Diferencia relativa:  {e['diferencia_relativa']:.2e}  -> Parseval OK: {e['parseval_ok']}")

Energía (tiempo):     3.9691e+08
Energía (frecuencia): 3.9691e+08
Diferencia relativa:  3.00e-16  -> Parseval OK: True


## `autocorrelacion(serie, nlags)`

ACF vía `statsmodels`. **Concepto:** autocorrelación; el pico en lag 7 confirma la
periodicidad semanal.

In [13]:
ac = utils.autocorrelacion(serie, nlags=21)
plt.figure(figsize=(12, 4)); plt.stem(range(len(ac)), ac)
plt.axvline(7, color="red", ls="--", lw=1, label="lag 7"); plt.legend()
plt.title("Autocorrelación (ACF)"); plt.show()
print(f"ACF en lag 7: {ac[7]:.3f}")

ACF en lag 7: 0.864


C:\Users\micag\AppData\Local\Temp\ipykernel_2384\11375462.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Autocorrelación (ACF)"); plt.show()


## `correlacion_cruzada(serie1, serie2, max_lag)`

Correlación cruzada normalizada por lag. **Concepto:** correlación cruzada entre dos
señales / detección de desfasaje.

In [14]:
serie_desf = serie.shift(3).bfill()  # misma serie corrida 3 días
cc = utils.correlacion_cruzada(serie, serie_desf, max_lag=14)
lag_max = cc.idxmax()
print(f"Lag de máxima correlación: {lag_max} (esperado ~3)")
plt.figure(figsize=(10, 4)); plt.stem(cc.index, cc.values)
plt.axvline(0, color="gray", lw=0.8); plt.title("Correlación cruzada"); plt.show()

Lag de máxima correlación: 3 (esperado ~3)


C:\Users\micag\AppData\Local\Temp\ipykernel_2384\702960604.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.axvline(0, color="gray", lw=0.8); plt.title("Correlación cruzada"); plt.show()


## `test_estacionariedad(serie)`

Corre ADF y KPSS (`statsmodels.tsa.stattools`). **Concepto:** estacionariedad de la
serie (requisito previo al modelado ARIMA).

In [15]:
res = utils.test_estacionariedad(serie)
print("ADF :", res["adf"])
print("KPSS:", res["kpss"])

ADF : {'estadistico': -0.09155131933205975, 'p_valor': 0.9502735797463145, 'estacionaria': np.False_}
KPSS: {'estadistico': 2.299690629848704, 'p_valor': 0.01, 'estacionaria': np.False_}


## `ajustar_arima(serie, order)`

Ajusta un ARIMA(p,d,q) y devuelve el modelo con su AIC/BIC. **Concepto:** modelado
paramétrico de la serie para pronóstico.

In [16]:
arima = utils.ajustar_arima(serie, order=(1, 1, 1))
print(f"AIC: {arima['aic']:.1f} | BIC: {arima['bic']:.1f}")

AIC: 2379.4 | BIC: 2389.0


## `ajustar_prophet(serie)`

Ajusta Prophet (tendencia + estacionalidad). **Concepto:** modelo aditivo de
descomposición para forecasting. *(Requiere el paquete `prophet`; si no está
instalado se omite.)*

In [17]:
try:
    modelo_p = utils.ajustar_prophet(serie)
    print("Prophet ajustado:", type(modelo_p).__name__)
except Exception as ex:
    print(f"Prophet no disponible, se omite: {type(ex).__name__}: {ex}")

Prophet no disponible, se omite: ModuleNotFoundError: No module named 'prophet'


## `backtesting_forecast(serie, modelo_fn, ventana_train, horizonte)`

Backtesting walk-forward: desliza la ventana de entrenamiento y acumula el error de
pronóstico. **Concepto:** validación fuera de muestra del modelo. Acá usamos un
pronóstico de persistencia (último valor) como `modelo_fn` de ejemplo.

In [18]:
def persistencia(train, h):
    return np.repeat(train.iloc[-1], h)

bt = utils.backtesting_forecast(serie, persistencia, ventana_train=60, horizonte=7)
print(f"MAE: {bt['mae']:.1f} | RMSE: {bt['rmse']:.1f} | MAPE: {bt['mape']:.1f}%")
print(f"Cantidad de errores acumulados: {len(bt['errores'])}")

MAE: 225.4 | RMSE: 272.5 | MAPE: 15.9%
Cantidad de errores acumulados: 119
